# Day 31 — Python Classes for Analytics Workflows
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Design DataLoader and QueryRunner classes with proper encapsulation
2. Use context managers for safe database connection management
3. Understand when OOP patterns add value in analytics vs when they don't

---
> **When OOP helps in analytics:** OOP is most valuable for *pipelines* (repeated operations on varying inputs) and *shared infrastructure* (connection management, config handling). It adds little value for one-off exploratory analysis where a simple script is clearer. In a consulting analytics context: your DataLoader class would be shared across 5 analysts on the same engagement — that's the right motivation for a class.


## 1. DataLoader Class

In [ ]:
from pathlib import Path
from typing import Optional
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

class DataLoader:
    """Manages loading and caching of CSV datasets for an analytics project.

    Attributes:
        data_path: Directory containing the CSV files.
        _cache: In-memory cache of loaded DataFrames.
    """

    KNOWN_FILES = {
        'materials': 'materials_inventory.csv',
        'sales_orders': 'sales_orders.csv',
        'cost_centers': 'cost_center_actuals.csv',
        'hr': 'hr_headcount.csv',
        'bw_kpis': 'bw_sales_kpis.csv',
    }

    def __init__(self, data_path: str | Path = '../datasets') -> None:
        self.data_path = Path(data_path)
        self._cache: dict[str, pd.DataFrame] = {}
        self.logger = logging.getLogger(self.__class__.__name__)

    def load(self, name: str, force_reload: bool = False, **kwargs) -> pd.DataFrame:
        """Load a named dataset, using cache unless force_reload=True.

        Args:
            name: Dataset name key (e.g., 'sales_orders').
            force_reload: If True, bypass cache and reload from disk.
            **kwargs: Additional kwargs passed to pd.read_csv().

        Returns:
            Loaded DataFrame.

        Raises:
            KeyError: If name is not a known dataset.
            FileNotFoundError: If the file does not exist.
        """
        # YOUR SOLUTION
        if name not in self.KNOWN_FILES:
            raise KeyError(f'Unknown dataset: {name!r}. Known: {list(self.KNOWN_FILES)}')

        if name in self._cache and not force_reload:
            self.logger.debug('Cache hit: %s', name)
            return self._cache[name]

        path = self.data_path / self.KNOWN_FILES[name]
        if not path.exists():
            raise FileNotFoundError(f'File not found: {path}')

        df = pd.read_csv(path, **kwargs)
        self._cache[name] = df
        self.logger.info('Loaded %s: %d rows', name, len(df))
        return df

    def load_all(self) -> dict[str, pd.DataFrame]:
        """Load all known datasets into cache and return them."""
        # YOUR SOLUTION
        return {name: self.load(name) for name in self.KNOWN_FILES}

    def clear_cache(self) -> None:
        """Clear the in-memory cache to free memory."""
        self._cache.clear()
        self.logger.info('Cache cleared.')

    @property
    def cached_datasets(self) -> list[str]:
        """Return list of currently cached dataset names."""
        return list(self._cache.keys())

    def __repr__(self) -> str:
        return f'DataLoader(data_path={self.data_path!r}, cached={self.cached_datasets})'

# Test DataLoader
loader = DataLoader('../datasets')
so = loader.load('sales_orders', parse_dates=['ERDAT'])
hr = loader.load('hr', parse_dates=['HIRE_DATE', 'TERM_DATE'])
print(loader)
print('Sales orders shape:', so.shape)

## 2. QueryRunner Class + Context Manager for DB Connections

In [ ]:
import sqlite3
from typing import Any

class DBConnection:
    """Context manager for SQLite database connections.

    Usage:
        with DBConnection(':memory:') as con:
            df = pd.read_sql_query('SELECT ...', con)

    Ensures the connection is always closed, even if an exception occurs.
    """

    def __init__(self, db_path: str) -> None:
        self.db_path = db_path
        self._con: Optional[sqlite3.Connection] = None

    def __enter__(self) -> sqlite3.Connection:
        self._con = sqlite3.connect(self.db_path)
        return self._con

    def __exit__(self, exc_type, exc_val, exc_tb) -> bool:
        if self._con:
            if exc_type is None:
                self._con.commit()
            else:
                self._con.rollback()
            self._con.close()
        return False  # don't suppress exceptions


class QueryRunner:
    """Runs SQL queries against a SQLite database, with optional result caching."""

    def __init__(self, db_path: str = ':memory:') -> None:
        self.db_path = db_path
        self._query_log: list[dict] = []
        self.logger = logging.getLogger(self.__class__.__name__)

    def load_dataframes(self, dfs: dict[str, pd.DataFrame]) -> None:
        """Load a dict of DataFrames into the SQLite DB as tables."""
        with DBConnection(self.db_path) as con:
            for name, df in dfs.items():
                df.to_sql(name, con, if_exists='replace', index=False)
                self.logger.info('Loaded table: %s (%d rows)', name, len(df))

    def run(self, sql: str, params: tuple = ()) -> pd.DataFrame:
        """Execute a SQL query and return results as a DataFrame."""
        import time
        start = time.time()
        with DBConnection(self.db_path) as con:
            df = pd.read_sql_query(sql, con, params=params)
        elapsed = time.time() - start
        self._query_log.append({'sql': sql[:60] + '...', 'rows': len(df), 'time_ms': round(elapsed*1000, 1)})
        self.logger.debug('Query returned %d rows in %.1f ms', len(df), elapsed*1000)
        return df

    def query_log(self) -> pd.DataFrame:
        """Return the query execution log as a DataFrame."""
        return pd.DataFrame(self._query_log)

# Test QueryRunner
all_dfs = loader.load_all()
runner = QueryRunner(':memory:')
runner.load_dataframes(all_dfs)

result = runner.run('''
    SELECT VKORG, COUNT(*) AS orders, ROUND(SUM(NETWR),0) AS total_value
    FROM sales_orders
    WHERE STATUS = 'OPEN'
    GROUP BY VKORG ORDER BY total_value DESC
''')
print(result)
print()
print('Query Log:')
print(runner.query_log())

## 3. Dataclass for Typed Report Config

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class ReportConfig:
    """Configuration for a parameterised analytics report.

    All fields are validated on creation via __post_init__.
    """
    period: int                        # YYYYMM format
    plant: Optional[str] = None        # SAP Plant code, None = all plants
    output_dir: Path = Path('.')
    include_charts: bool = True
    export_formats: list[str] = field(default_factory=lambda: ['csv', 'html'])
    analyst_name: str = 'Unknown'

    def __post_init__(self) -> None:
        # Validate period format
        year, month = self.period // 100, self.period % 100
        if not (2000 <= year <= 2099):
            raise ValueError(f'Invalid period year: {year}')
        if not (1 <= month <= 12):
            raise ValueError(f'Invalid period month: {month}')
        # Validate export formats
        valid_fmts = {'csv', 'html', 'xlsx'}
        invalid = set(self.export_formats) - valid_fmts
        if invalid:
            raise ValueError(f'Invalid export format(s): {invalid}')

    @property
    def period_label(self) -> str:
        """Return human-readable period label, e.g. 'December 2024'."""
        import calendar
        year, month = self.period // 100, self.period % 100
        return f'{calendar.month_name[month]} {year}'

    def __repr__(self) -> str:
        return f'ReportConfig(period={self.period_label}, plant={self.plant!r})'


# Test dataclass
cfg = ReportConfig(period=202412, plant='1000', analyst_name='Jane Smith')
print(cfg)
print('Period label:', cfg.period_label)

# Validation test
try:
    bad = ReportConfig(period=202413)  # invalid month
except ValueError as e:
    print(f'Caught expected error: {e}')

## 4. When OOP Helps vs Hurts in Analytics

### OOP Guidance for Analytics

| Situation | Use OOP? | Why |
|---|---|---|
| One-off EDA in a notebook | No | Functions are simpler and sufficient |
| Shared data pipeline used by 3+ analysts | Yes | Encapsulation prevents inconsistency |
| DB connection management | Yes | Context managers enforce cleanup |
| Single SQL query | No | Just write `pd.read_sql_query(...)` |
| Complex multi-step report with parameters | Yes | Config dataclass makes it testable |
| Visualisation helper | Maybe | A function is usually enough |

**Rule of thumb:** If you find yourself copying and pasting the same 10 lines across 5 notebooks, that's a class.


In [ ]:
# YOUR SOLUTION
# Put it all together: Use DataLoader + QueryRunner + ReportConfig to
# answer: "What is the total open order backlog for plant 1000 as of period 202412?"

config = ReportConfig(period=202412, plant='1000')
print(f'Running report: {config}')

# Load data
dl = DataLoader('../datasets')
all_data = dl.load_all()

# Run query
qr = QueryRunner(':memory:')
qr.load_dataframes(all_data)

# The QueryRunner + ReportConfig combo: query is parameterised by config
backlog_sql = '''
    SELECT s.MATNR, m.WERKS, m.MAKTX,
           COUNT(*) AS open_lines,
           ROUND(SUM(s.NETWR), 0) AS backlog_value
    FROM sales_orders s
    JOIN materials m ON s.MATNR = m.MATNR
    WHERE s.STATUS = 'OPEN'
      AND m.WERKS = ?
    GROUP BY s.MATNR, m.WERKS, m.MAKTX
    ORDER BY backlog_value DESC
    LIMIT 10
'''

result = qr.run(backlog_sql, params=(config.plant,))
print(f'\nTop 10 open order materials — Plant {config.plant}:')
print(result.to_string(index=False))

## Daily Prompt
**Answer independently:**

1. Add a `validate()` method to `DataLoader` that checks all 5 CSV files exist and returns a summary dict: `{'materials': True, 'hr': True, ...}`.
2. Extend `QueryRunner` to support named parameters in SQL using a dict (e.g., `params={'status': 'OPEN'}`). How does SQLite handle named params vs positional `?`?
3. Write a `CostCenterReport` dataclass that inherits from `ReportConfig` and adds: `cost_center_list: list[str]` and a `fiscal_year` field derived from `period`. Override `__repr__`.
